# 🔐 Log Analytics Workspace - Resource-Context Access Control Test

Dieses Notebook demonstriert die **Resource-Context Access Control** in Azure Log Analytics:
- Zwei Service Principals (SP1 = User1, SP2 = User2)
- Jeder SP hat nur Reader-Zugriff auf seinen eigenen Storage Account
- Der zentrale LAW ist auf `enableLogAccessUsingOnlyResourcePermissions = true` konfiguriert
- **Ergebnis**: Jeder User sieht nur die Logs seiner eigenen Ressourcen!

## Multi-Subscription Architektur

```
┌─────────────────────────────────────────────────────────────────────────────┐
│              Central Subscription: onprem (4b353dc5-...)                    │
│  ┌───────────────────────────────────────────────────────────────────────┐  │
│  │                    law-central-shared                                  │  │
│  │           (enableLogAccessUsingOnlyResourcePermissions = true)        │  │
│  │                                                                        │  │
│  │   ┌────────────────────────┐      ┌────────────────────────┐          │  │
│  │   │ Logs: cptdazmonlog-    │      │ Logs: cptdazmonlog-    │          │  │
│  │   │       access1          │      │       access2          │          │  │
│  │   │ → Nur für SP1 sichtbar │      │ → Nur für SP2 sichtbar │          │  │
│  │   └───────────▲────────────┘      └───────────▲────────────┘          │  │
│  └───────────────┼───────────────────────────────┼────────────────────────┘  │
└──────────────────┼───────────────────────────────┼──────────────────────────┘
                   │                               │
     ┌─────────────┴─────────────┐   ┌─────────────┴─────────────┐
     │  sub-cptdx-lz1            │   │  sub-cptdx-lz2            │
     │  (d3856ecd-...)           │   │  (a2745aeb-...)           │
     │  ┌─────────────────────┐  │   │  ┌─────────────────────┐  │
     │  │ cptdazmonlogaccess1 │  │   │  │ cptdazmonlogaccess2 │  │
     │  │ rg-user1-resources  │  │   │  │ rg-user2-resources  │  │
     │  │ + Diagnostic        │  │   │  │ + Diagnostic        │  │
     │  │   Settings → LAW    │  │   │  │   Settings → LAW    │  │
     │  └─────────────────────┘  │   │  └─────────────────────┘  │
     │                           │   │                           │
     │  SP1: Reader + Blob       │   │  SP2: Reader + Blob       │
     │       Data Contributor    │   │       Data Contributor    │
     └───────────────────────────┘   └───────────────────────────┘
```

## RBAC Übersicht

| Principal | Storage Account | Rollen |
|-----------|-----------------|--------|
| SP1 (000808e6-...) | cptdazmonlogaccess1 | Reader, Storage Blob Data Contributor |
| SP2 (cf37bbf0-...) | cptdazmonlogaccess2 | Reader, Storage Blob Data Contributor |
| Portal User1 (e1328847-...) | cptdazmonlogaccess1 | Reader |
| Portal User2 (11768f04-...) | cptdazmonlogaccess2 | Reader |

## 1. Setup - Helper Functions und Konfiguration

In [ ]:
# ============================================================================
# Helper Functions und Konfiguration
# ============================================================================

import os
import re
import subprocess
import json
import requests

# Pfade
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
TERRAFORM_DIR = os.path.join(NOTEBOOK_DIR, "terraform")
TFVARS_PATH = os.path.join(TERRAFORM_DIR, "terraform.tfvars")

# Separate Azure CLI Config-Verzeichnisse für isolierte Kontexte
SP1_CONFIG_DIR = os.path.join(TERRAFORM_DIR, ".azure-sp1")
SP2_CONFIG_DIR = os.path.join(TERRAFORM_DIR, ".azure-sp2")

def run(cmd, capture_output=True):
    """Führe einen Shell-Befehl aus und gib das Ergebnis zurück"""
    try:
        result = subprocess.run(
            cmd, 
            shell=True, 
            capture_output=capture_output,
            text=True,
            env=os.environ.copy()
        )
        if capture_output:
            if result.returncode == 0:
                return result.stdout.strip()
            else:
                return f"ERROR: {result.stderr.strip()}"
        return result.returncode
    except Exception as e:
        return f"EXCEPTION: {str(e)}"

def run_json(cmd):
    """Führe einen az CLI Befehl aus und parse JSON output"""
    result = run(cmd)
    if result and not result.startswith("ERROR") and not result.startswith("EXCEPTION"):
        try:
            return json.loads(result)
        except json.JSONDecodeError:
            return None
    return None

def query_logs(test_name, token, storage_id, query="StorageBlobLogs | take 2"):
    """
    Zentrale Funktion zum Abfragen von Logs via Log Analytics API.
    Zeigt nur die Rows im Output an.
    
    Returns: tuple (rows, success)
        - rows: Liste der Zeilen oder None bei Fehler
        - success: True wenn Daten vorhanden, False sonst
    """
    url = f"https://api.loganalytics.io/v1{storage_id}/query"
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    
    print("=" * 60)
    print(f"TEST: {test_name}")
    print("=" * 60)
    print(f"Storage: ...{storage_id.split('/')[-1]}")
    print(f"Query: {query}")
    print()
    
    try:
        response = requests.post(url, headers=headers, json={"query": query}, timeout=30)
    except Exception as e:
        print(f"❌ Request fehlgeschlagen: {e}")
        return None, False
    
    print(f"HTTP Status: {response.status_code}")
    print()
    
    if response.status_code == 200:
        data = response.json()
        if data.get("tables") and len(data["tables"]) > 0:
            table = data["tables"][0]
            columns = [col["name"] for col in table.get("columns", [])]
            rows = table.get("rows", [])
            
            print(f"Columns: {columns}")
            print(f"Rows ({len(rows)}):")
            print("-" * 60)
            for i, row in enumerate(rows):
                print(f"[{i}] {row}")
            print("-" * 60)
            
            return rows, len(rows) > 0
        else:
            print("⚠️ Keine Tabellen in der Antwort")
            return [], False
    else:
        print(f"❌ Fehler: {response.status_code}")
        try:
            error_data = response.json()
            print(f"   {error_data.get('error', {}).get('message', response.text)}")
        except:
            print(f"   {response.text[:200]}")
        return None, False

def parse_tfvars(file_path):
    """Parse terraform.tfvars und extrahiere Key-Value Paare"""
    values = {}
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#'):
                    match = re.match(r'(\w+)\s*=\s*"([^"]*)"', line)
                    if match:
                        values[match.group(1)] = match.group(2)
    return values

def get_terraform_output():
    """Lade Terraform Output als Dictionary"""
    original_dir = os.getcwd()
    os.chdir(TERRAFORM_DIR)
    result = subprocess.run(["terraform", "output", "-json"], capture_output=True, text=True)
    os.chdir(original_dir)
    if result.returncode == 0:
        return json.loads(result.stdout)
    return {}

# ============================================================================
# Lade Konfiguration aus terraform.tfvars und terraform output
# ============================================================================

print(f"📂 Lade Konfiguration...")
config = parse_tfvars(TFVARS_PATH)
tf_output = get_terraform_output()

# Extrahiere Werte aus tfvars
TENANT_ID = config.get("tenant_id", "")
SP1_CLIENT_ID = config.get("sp1_client_id", "")
SP1_CLIENT_SECRET = config.get("sp1_client_secret", "")
SP2_CLIENT_ID = config.get("sp2_client_id", "")
SP2_CLIENT_SECRET = config.get("sp2_client_secret", "")

# Extrahiere Werte aus Terraform Output
user1_res = tf_output.get("user1_resources", {}).get("value", {})
user2_res = tf_output.get("user2_resources", {}).get("value", {})
central_law = tf_output.get("central_law", {}).get("value", {})

SP1_STORAGE = user1_res.get("storage_account_name", "")
SP1_STORAGE_ID = user1_res.get("storage_account_id", "")
SP1_SUBSCRIPTION_ID = user1_res.get("subscription_id", "")

SP2_STORAGE = user2_res.get("storage_account_name", "")
SP2_STORAGE_ID = user2_res.get("storage_account_id", "")
SP2_SUBSCRIPTION_ID = user2_res.get("subscription_id", "")

LAW_WORKSPACE_ID = central_law.get("workspace_id", "")

print(f"✅ Konfiguration geladen:")
print(f"   Tenant ID: {TENANT_ID[:8]}...{TENANT_ID[-4:] if len(TENANT_ID) > 8 else ''}")
print(f"   LAW Workspace ID: {LAW_WORKSPACE_ID}")
print(f"   SP1 Client ID: {SP1_CLIENT_ID[:8]}...{SP1_CLIENT_ID[-4:] if len(SP1_CLIENT_ID) > 8 else ''}")
print(f"   SP2 Client ID: {SP2_CLIENT_ID[:8]}...{SP2_CLIENT_ID[-4:] if len(SP2_CLIENT_ID) > 8 else ''}")
print(f"   SP1 Storage: {SP1_STORAGE} (Sub: {SP1_SUBSCRIPTION_ID[:8]}...)")
print(f"   SP2 Storage: {SP2_STORAGE} (Sub: {SP2_SUBSCRIPTION_ID[:8]}...)")

## 2. Authentifizierung als Service Principal 1 (User1)

Wir erstellen ein **isoliertes Azure CLI Config-Verzeichnis** für SP1, damit der Login den persönlichen Kontext nicht beeinflusst.

## 1.5 Data Plane Operationen mit Service Principals

Jeder SP erstellt Container und Blobs **nur auf seinem eigenen Storage Account**.
Dies demonstriert, dass Data Plane Operationen via RBAC (Storage Blob Data Contributor) funktionieren.

In [ ]:
# ============================================================================
# Data Plane Operationen mit Service Principals
# ============================================================================
# Jeder SP erstellt Container und Blobs nur auf seinem eigenen Storage Account
# Dies erfordert "Storage Blob Data Contributor" Rolle (via Terraform zugewiesen)
# ============================================================================

import tempfile
import datetime

def execute_data_plane_operations(sp_name, client_id, client_secret, tenant_id, storage_account, config_dir):
    """
    Führt Data Plane Operationen für einen Service Principal aus.
    Returns: dict mit Ergebnissen (success, blobs, errors)
    """
    results = {"success": True, "blobs": [], "errors": []}
    container_name = "testlogs"
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    
    print(f"🔐 {sp_name} Login für Data Plane Operationen...")
    os.makedirs(config_dir, exist_ok=True)
    os.environ["AZURE_CONFIG_DIR"] = config_dir
    
    # Login - verwende -p=secret Syntax für Passwörter die mit - beginnen
    login_result = run(f'az login --service-principal -u "{client_id}" -p="{client_secret}" --tenant "{tenant_id}" --output none', capture_output=False)
    
    if login_result != 0:
        results["success"] = False
        results["errors"].append("Login fehlgeschlagen")
        print(f"❌ {sp_name} Login fehlgeschlagen!")
        return results
    
    print(f"✅ {sp_name} eingeloggt")
    
    # Container erstellen
    result = run(f'az storage container create --name {container_name} --account-name {storage_account} --auth-mode login 2>&1')
    if "ERROR" in result:
        results["errors"].append(f"Container: {result}")
        print(f"   ❌ Container '{container_name}': FEHLER - {result[:100]}")
    else:
        print(f"   📁 Container '{container_name}' auf {storage_account}: erstellt/existiert")
    
    # Blob hochladen
    blob_name = f"{sp_name.lower()}_test_{timestamp}.txt"
    blob_content = f"{sp_name} blob created at {timestamp}"
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
        f.write(blob_content)
        temp_file = f.name
    
    result = run(f'az storage blob upload --container-name {container_name} --name {blob_name} --file "{temp_file}" --account-name {storage_account} --auth-mode login --overwrite 2>&1')
    if "ERROR" in result:
        results["success"] = False
        results["errors"].append(f"Upload: {result}")
        print(f"   ❌ Blob '{blob_name}': FEHLER - {result[:100]}")
    else:
        print(f"   📄 Blob '{blob_name}': hochgeladen")
    
    # Blob Liste anzeigen
    result = run(f'az storage blob list --container-name {container_name} --account-name {storage_account} --auth-mode login --query "[].name" -o tsv 2>&1')
    print(f"   📖 Blob-Liste:")
    if result and "ERROR" not in result:
        for blob in result.split('\n'):
            if blob.strip():
                results["blobs"].append(blob.strip())
                print(f"      - {blob.strip()}")
    else:
        results["errors"].append(f"List: {result}")
        print(f"      ⚠️ Fehler beim Auflisten: {result[:100]}")
    
    return results

# ============================================================================
# Ausführung für beide SPs
# ============================================================================

print("=" * 60)
print("📦 DATA PLANE OPERATIONEN MIT SERVICE PRINCIPALS")
print("=" * 60)
print()

# SP1 Operationen
sp1_results = execute_data_plane_operations(
    sp_name="SP1",
    client_id=SP1_CLIENT_ID,
    client_secret=SP1_CLIENT_SECRET,
    tenant_id=TENANT_ID,
    storage_account=SP1_STORAGE,
    config_dir=SP1_CONFIG_DIR
)
print()

# SP2 Operationen
sp2_results = execute_data_plane_operations(
    sp_name="SP2",
    client_id=SP2_CLIENT_ID,
    client_secret=SP2_CLIENT_SECRET,
    tenant_id=TENANT_ID,
    storage_account=SP2_STORAGE,
    config_dir=SP2_CONFIG_DIR
)

# Zusammenfassung
print()
print("=" * 60)
print("📊 ZUSAMMENFASSUNG DATA PLANE OPERATIONEN")
print("=" * 60)
print(f"SP1 ({SP1_STORAGE}): {'✅ Erfolgreich' if sp1_results['success'] else '❌ Fehler'}")
if sp1_results["errors"]:
    for err in sp1_results["errors"]:
        print(f"   ⚠️ {err[:80]}")
print(f"SP2 ({SP2_STORAGE}): {'✅ Erfolgreich' if sp2_results['success'] else '❌ Fehler'}")
if sp2_results["errors"]:
    for err in sp2_results["errors"]:
        print(f"   ⚠️ {err[:80]}")
print("=" * 60)
print("⏳ Logs erscheinen in 5-10 Minuten im LAW")

In [ ]:
# ============================================================================
# Login als Service Principal 1 (User1)
# ============================================================================

# Erstelle isoliertes Config-Verzeichnis für SP1
os.makedirs(SP1_CONFIG_DIR, exist_ok=True)
os.environ["AZURE_CONFIG_DIR"] = SP1_CONFIG_DIR

print(f"🔒 Isolierter Azure CLI Kontext für SP1:")
print(f"   Config-Verzeichnis: {SP1_CONFIG_DIR}")
print()

# Login mit SP1 - verwende -p=secret Syntax für Passwörter die mit - beginnen
print("🔐 Logging in als Service Principal 1 (User1)...")
login_result = run(f'az login --service-principal -u "{SP1_CLIENT_ID}" -p="{SP1_CLIENT_SECRET}" --tenant "{TENANT_ID}" --output none', capture_output=False)

if login_result == 0:
    print("✅ SP1 Login erfolgreich!")
    
    # Zeige Account Details
    account = run_json("az account show")
    if account:
        print(f"   User: {account.get('user', {}).get('name', 'N/A')}")
        print(f"   Tenant: {account.get('tenantId', 'N/A')[:8]}...")
else:
    print("❌ SP1 Login fehlgeschlagen!")

## 3. Logs abfragen als SP1 (User1)

SP1 hat nur Reader-Zugriff auf Storage1. Er sollte **nur Logs von diesem Storage Account** sehen.

In [ ]:
# Hole Access Token für SP1
import requests

os.environ["AZURE_CONFIG_DIR"] = SP1_CONFIG_DIR

# Resource IDs werden aus Zelle 3 übernommen (SP1_STORAGE_ID, SP2_STORAGE_ID)
print(f"SP1 Storage ID: {SP1_STORAGE_ID}")

token_result = run('az account get-access-token --resource https://api.loganalytics.io -o json')
if token_result and not token_result.startswith("ERROR"):
    sp1_token = json.loads(token_result).get("accessToken", "")
    print("✅ Access Token für SP1 erhalten")
else:
    print(f"❌ Token-Abruf fehlgeschlagen")
    sp1_token = ""

In [ ]:
# TEST 1: SP1 queried EIGENEN Storage Account
sp1_own_logs, sp1_own_success = query_logs(
    test_name="SP1 → eigener Storage Account",
    token=sp1_token,
    storage_id=SP1_STORAGE_ID
)

In [ ]:
# TEST 2: SP1 versucht SP2's Storage zu querien (sollte scheitern)
_, sp1_can_see_sp2 = query_logs(
    test_name="SP1 → SP2's Storage (sollte scheitern)",
    token=sp1_token,
    storage_id=SP2_STORAGE_ID
)

## 4. Authentifizierung als Service Principal 2 (User2)

Jetzt wechseln wir zu einem **separaten Azure CLI Kontext** für SP2.

In [ ]:
# ============================================================================
# Login als Service Principal 2 (User2)
# ============================================================================

# Erstelle isoliertes Config-Verzeichnis für SP2
os.makedirs(SP2_CONFIG_DIR, exist_ok=True)
os.environ["AZURE_CONFIG_DIR"] = SP2_CONFIG_DIR

print(f"🔒 Isolierter Azure CLI Kontext für SP2:")
print(f"   Config-Verzeichnis: {SP2_CONFIG_DIR}")
print()

# Login mit SP2 - verwende -p=secret Syntax für Passwörter die mit - beginnen
print("🔐 Logging in als Service Principal 2 (User2)...")
login_result = run(f'az login --service-principal -u "{SP2_CLIENT_ID}" -p="{SP2_CLIENT_SECRET}" --tenant "{TENANT_ID}" --output none', capture_output=False)

if login_result == 0:
    print("✅ SP2 Login erfolgreich!")
    
    # Zeige Account Details
    account = run_json("az account show")
    if account:
        print(f"   User: {account.get('user', {}).get('name', 'N/A')}")
        print(f"   Tenant: {account.get('tenantId', 'N/A')[:8]}...")
else:
    print("❌ SP2 Login fehlgeschlagen!")

## 5. Logs abfragen als SP2 (User2)

SP2 hat nur Reader-Zugriff auf Storage2. Er sollte **nur Logs von diesem Storage Account** sehen.

In [ ]:
# Hole Access Token für SP2
os.environ["AZURE_CONFIG_DIR"] = SP2_CONFIG_DIR

token_result = run('az account get-access-token --resource https://api.loganalytics.io -o json')
if token_result and not token_result.startswith("ERROR"):
    token_data = json.loads(token_result)
    sp2_token = token_data.get("accessToken", "")
    print("✅ Access Token für SP2 erhalten")
else:
    print(f"❌ Token-Abruf fehlgeschlagen: {token_result}")
    sp2_token = ""

In [ ]:
# TEST 3: SP2 queried EIGENEN Storage Account
sp2_own_logs, sp2_own_success = query_logs(
    test_name="SP2 → eigener Storage Account",
    token=sp2_token,
    storage_id=SP2_STORAGE_ID
)

In [ ]:
# TEST 4: SP2 versucht SP1's Storage zu querien (sollte scheitern)
_, sp2_can_see_sp1 = query_logs(
    test_name="SP2 → SP1's Storage (sollte scheitern)",
    token=sp2_token,
    storage_id=SP1_STORAGE_ID
)

## 6. Beweis der Zugriffsisolation

Hier vergleichen wir, welche Ressourcen jeder SP sehen kann. 
- **SP1 sollte NUR `stuser1mh53kg` sehen**
- **SP2 sollte NUR `stuser2mh53kg` sehen**
- **Kein SP sollte die Logs des anderen sehen können!**

In [ ]:
# ZUSAMMENFASSUNG
print("=" * 60)
print("🔍 RESOURCE-CONTEXT ACCESS CONTROL - ERGEBNIS")
print("=" * 60)

print(f"\nSP1 ({SP1_STORAGE}):")
print(f"  • Eigene Logs sehen: {'✅ JA' if sp1_own_success else '⚠️ NEIN'}")
print(f"  • Fremde Logs sehen: {'❌ JA (Problem!)' if sp1_can_see_sp2 else '✅ NEIN (korrekt)'}")

print(f"\nSP2 ({SP2_STORAGE}):")
print(f"  • Eigene Logs sehen: {'✅ JA' if sp2_own_success else '⚠️ NEIN'}")
print(f"  • Fremde Logs sehen: {'❌ JA (Problem!)' if sp2_can_see_sp1 else '✅ NEIN (korrekt)'}")

print("\n" + "=" * 60)
if not sp1_can_see_sp2 and not sp2_can_see_sp1:
    print("🎉 ACCESS CONTROL FUNKTIONIERT!")
else:
    print("⚠️ WARNUNG: Zugriffsisolation nicht korrekt!")
print("=" * 60)